<div style="text-align:center;">
  <b>Национальный исследовательский университет ИТМО</b><br>
  Факультет информационных технологий и программирования<br>
  Кафедра Компьютерных Технологий
</div>


---

# **Методы оптимизации**  
### Лабораторная работа №3 
### Метод стохастического градиентного спуска (SGD) и его модификации

<div style="text-align:right; font-size:12px">
Выполнили:<br>
Алфёров Кирилл М3232<br>
Салов Егор М3232
</div>





### Информация о датасете

Датасет содержит 9358 записей с почасовыми усреднёнными откликами от массива из 5 химических сенсоров на основе металлических оксидов, встроенных в устройство химического мониторинга качества воздуха. Устройство было размещено на улице в зоне с высоким уровнем загрязнения, на уровне дороги, в одном из городов Италии. Данные собирались с марта 2004 по февраль 2005 года (в течение одного года), что делает этот набор данных одним из самых продолжительных по длительности свободно доступных полевых наблюдений с использованием химических сенсоров.

Эталонные (ground truth) данные о почасовых усреднённых концентрациях угарного газа (CO), неметановых углеводородов, бензола, суммарных оксидов азота (NOx) и диоксида азота (NO₂) были получены с помощью сертифицированного анализатора, установленного рядом.

В данных присутствуют признаки перекрёстной чувствительности сенсоров, а также концептуального и сенсорного дрейфа, что описано в статье De Vito и др., Sens. and Act. B, Vol. 129, №2, 2008 (необходима ссылка на источник). Эти эффекты могут снижать точность оценки концентраций загрязняющих веществ по данным сенсоров. Пропущенные значения помечены как -200.

### Читаем датасет

In [14]:
import pandas

dataset = pandas.read_csv("dataset/AirQualityUCI.csv", sep=';', decimal=',')
dataset

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,2.6,1360.0,150.0,11.9,1046.0,166.0,1056.0,113.0,1692.0,1268.0,13.6,48.9,0.7578,NaN,NaN
1,10/03/2004,19.00.00,2.0,1292.0,112.0,9.4,955.0,103.0,1174.0,92.0,1559.0,972.0,13.3,47.7,0.7255,NaN,NaN
2,10/03/2004,20.00.00,2.2,1402.0,88.0,9.0,939.0,131.0,1140.0,114.0,1555.0,1074.0,11.9,54.0,0.7502,NaN,NaN
3,10/03/2004,21.00.00,2.2,1376.0,80.0,9.2,948.0,172.0,1092.0,122.0,1584.0,1203.0,11.0,60.0,0.7867,NaN,NaN
4,10/03/2004,22.00.00,1.6,1272.0,51.0,6.5,836.0,131.0,1205.0,116.0,1490.0,1110.0,11.2,59.6,0.7888,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9466,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9468,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9469,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Вычислим корреляцию параметров попарно

In [15]:
dataset.corr(numeric_only=True)

,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
CO(GT),1.000000,0.041411,0.128351,-0.031378,0.029926,0.526451,-0.089981,0.671127,-0.073724,0.080310,-0.068939,-0.048227,-0.045892,NaN,NaN
PT08.S1(CO),0.041411,1.000000,0.170007,0.852687,0.933102,0.277993,0.087019,0.154030,0.845149,0.892434,0.754844,0.745375,0.764903,NaN,NaN
NMHC(GT),0.128351,0.170007,1.000000,0.037323,0.110104,-0.004427,0.048821,0.103307,0.162680,0.101185,-0.000009,0.008284,0.012500,NaN,NaN
C6H6(GT),-0.031378,0.852687,0.037323,1.000000,0.767433,-0.001174,0.512193,-0.010992,0.774673,0.641334,0.971375,0.925062,0.984555,NaN,NaN
PT08.S2(NMHC),0.029926,0.933102,0.110104,0.767433,1.000000,0.331272,-0.073667,0.176488,0.874782,0.909905,0.669025,0.585803,0.646572,NaN,NaN
NOx(GT),0.526451,0.277993,-0.004427,-0.001174,0.331272,1.000000,-0.436084,0.817139,0.035546,0.461889,-0.138452,-0.053009,-0.095847,NaN,NaN
PT08.S3(NOx),-0.089981,0.087019,0.048821,0.512193,-0.073667,-0.436084,1.000000,-0.256232,0.122734,-0.208865,0.588111,0.573549,0.621618,NaN,NaN
NO2(GT),0.671127,0.154030,0.103307,-0.010992,0.176488,0.817139,-0.256232,1.000000,-0.022174,0.253439,-0.084104,-0.081305,-0.060440,NaN,NaN
PT08.S4(NO2),-0.073724,0.845149,0.162680,0.774673,0.874782,0.035546,0.122734,-0.022174,1.000000,0.723690,0.755060,0.640707,0.691913,NaN,NaN
PT08.S5(O3),0.080310,0.892434,0.101185,0.641334,0.909905,0.461889,-0.208865,0.253439,0.723690,1.000000,0.503700,0.524955,0.519467,NaN,NaN


### Что берём?


Для нашей регрессионной модели мы решили предсказывать именно **C6H6(GT)** (концентрацию бензола, мкг/м³). Вот почему:

1. **Очень сильная линейная связь**  
   - С абсолютной влажностью (AH): *r* ≈ 0.985  
   - С температурой воздуха (T): *r* ≈ 0.971  
   - С относительной влажностью (RH): *r* ≈ 0.925  
   
   Когда |*r*| > 0.9, простая линейная модель уже способна выдавать точные и стабильные предсказания.

2. **Минимум избыточности среди признаков**  
   Мы исключили сенсоры, которые почти полностью дублировали друг друга (корреляция > 0.9 между ними). В результате в качестве входных факторов остались только самые информативные переменные без сильной взаимозависимости


Итак, зависимая переменная **y = C6H6(GT)**, независимые переменные **x = AH (|r| = 0.985), PT08.S1(CO) (|r| = 0.853), PT08.S3(NOx) (|r| = 0.512)** (а также у датчиков PT08.S1(CO) и PT08.S3(NOx) слабая корреляция |r| = 0.087, что снижает мультиколлинеарность)


